# Supplementary runtime and CO2 measurement, v4

This notebook runs the additive script `scripts/run_supplementary_runtime_co2_v4.py`.

The script measures one successful eligible document per method/dataset, counts the eligible documents in the JSONL file, and extrapolates the total time and CO2 indicators. It writes CSV/JSON outputs to `data/suplementary_metrics/`.

Compared with v2, this version stores compact error diagnostics and, by default, uses `--sample-strategy first-success`, which tries the next eligible document if the selected sample fails. Failed candidate samples are recorded but are not used for CO2 extrapolation.

In [2]:
%load_ext autoreload
%autoreload 2
# Run this notebook from tests/processing/. Move to the project root.
%cd ../..

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
/home/galencarmedeiro/git/postdoc/ragtree


In [3]:
from pathlib import Path
ROOT = Path.cwd()
ROOT

PosixPath('/home/galencarmedeiro/git/postdoc/ragtree')

## List configured method/dataset runs

In [4]:
%run scripts/run_supplementary_runtime_co2_v4.py --list

Configured method/dataset runs: 57
001. LLM-only               | baseline                   | docred_causal      | filter=dev      | mode=direct_llm
002. LLM-only               | baseline                   | eventstoryline     | filter=all      | mode=direct_llm
003. LLM-only               | baseline                   | fincausal          | filter=all      | mode=direct_llm
004. LLM-only               | baseline                   | maven_ere          | filter=all      | mode=direct_llm
005. LLM-only               | baseline                   | causalbank         | filter=all      | mode=direct_llm
006. LLM-only               | cot                        | docred_causal      | filter=dev      | mode=direct_llm
007. LLM-only               | cot                        | eventstoryline     | filter=all      | mode=direct_llm
008. LLM-only               | cot                        | fincausal          | filter=all      | mode=direct_llm
009. LLM-only               | cot                    

## Dry run

This resolves the commands and selected documents without executing the methods.

In [5]:
%run scripts/run_supplementary_runtime_co2_v4.py \
  --dry-run \
  --sample-index 0 \
  --sample-strategy first-success

[supplementary] root=/home/galencarmedeiro/git/postdoc/ragtree
[supplementary] source config=/home/galencarmedeiro/git/postdoc/ragtree/configs/default.yaml
[supplementary] runtime config=/home/galencarmedeiro/git/postdoc/ragtree/data/suplementary_metrics/runtime_measurement_config.yaml
[supplementary] output dir=/home/galencarmedeiro/git/postdoc/ragtree/data/suplementary_metrics
[supplementary] selected runs=57
[supplementary] power_kw=0.3
[supplementary] carbon_intensity_kg_per_kwh=0.045
[supplementary] sample_strategy=first-success
[supplementary] max_sample_attempts=20
[1/57] LLM-only | baseline | docred_causal | eligible=998 | sample_skip=0 | doc_id=DocRED - e37288ca6012859f
    -> DRY-RUN: command resolved but not executed.
[2/57] LLM-only | baseline | eventstoryline | eligible=443 | sample_skip=0 | doc_id=EventStoryLine - 1_10ecbplus
    -> DRY-RUN: command resolved but not executed.
[3/57] LLM-only | baseline | fincausal | eligible=967 | sample_skip=0 | doc_id=FinCausal - bfdb8b

## Single-run test

Use this before launching all 57 runs. If the first document fails, v4 will try subsequent eligible documents up to `--max-sample-attempts`.

In [7]:
%run scripts/run_supplementary_runtime_co2_v4.py \
  --only-method baseline \
  --only-dataset docred_causal \
  --sample-index 0 \
  --sample-strategy first-success \
  --max-sample-attempts 20 \
  --power-kw 0.300 \
  --carbon-intensity 0.045 \
  --show-error-tail 80

[supplementary] root=/home/galencarmedeiro/git/postdoc/ragtree
[supplementary] source config=/home/galencarmedeiro/git/postdoc/ragtree/configs/default.yaml
[supplementary] runtime config=/home/galencarmedeiro/git/postdoc/ragtree/data/suplementary_metrics/runtime_measurement_config.yaml
[supplementary] output dir=/home/galencarmedeiro/git/postdoc/ragtree/data/suplementary_metrics
[supplementary] selected runs=1
[supplementary] power_kw=0.3
[supplementary] carbon_intensity_kg_per_kwh=0.045
[supplementary] sample_strategy=first-success
[supplementary] max_sample_attempts=20
[supplementary] vLLM preflight reached http://localhost:8000/v1/models with status 200.
[1/1] LLM-only | baseline | docred_causal | eligible=998 | sample_skip=0 | doc_id=DocRED - e37288ca6012859f
    -> OK: elapsed=14.55s, estimated_total=242.09min, estimated_total_co2=54.4691g
[supplementary] wrote:
  - /home/galencarmedeiro/git/postdoc/ragtree/data/suplementary_metrics/runtime_co2_by_method_dataset.csv
  - /home/gale

## Run all method/dataset configurations

Run this after the single-run test succeeds.

In [8]:
%run scripts/run_supplementary_runtime_co2_v4.py \
  --sample-index 0 \
  --sample-strategy first-success \
  --max-sample-attempts 20 \
  --power-kw 0.300 \
  --carbon-intensity 0.045 \
  --show-error-tail 40

[supplementary] root=/home/galencarmedeiro/git/postdoc/ragtree
[supplementary] source config=/home/galencarmedeiro/git/postdoc/ragtree/configs/default.yaml
[supplementary] runtime config=/home/galencarmedeiro/git/postdoc/ragtree/data/suplementary_metrics/runtime_measurement_config.yaml
[supplementary] output dir=/home/galencarmedeiro/git/postdoc/ragtree/data/suplementary_metrics
[supplementary] selected runs=57
[supplementary] power_kw=0.3
[supplementary] carbon_intensity_kg_per_kwh=0.045
[supplementary] sample_strategy=first-success
[supplementary] max_sample_attempts=20
[supplementary] vLLM preflight reached http://localhost:8000/v1/models with status 200.
[1/57] LLM-only | baseline | docred_causal | eligible=998 | sample_skip=0 | doc_id=DocRED - e37288ca6012859f
    -> OK: elapsed=11.39s, estimated_total=189.39min, estimated_total_co2=42.6139g
[2/57] LLM-only | baseline | eventstoryline | eligible=443 | sample_skip=0 | doc_id=EventStoryLine - 1_10ecbplus
    -> OK: elapsed=14.94s, e

## Read the generated CSV

In [9]:
import pandas as pd
from pathlib import Path
path = Path('data/suplementary_metrics/runtime_co2_by_method_dataset.csv')
df = pd.read_csv(path)
df.head()

,status,dry_run,success,return_code,error,error_type,error_message,error_traceback_tail,method_family,method,...,elapsed_minutes_doc,energy_kwh_doc,co2_kg_doc,co2_g_doc,estimated_total_seconds,estimated_total_minutes,estimated_total_hours,estimated_total_energy_kwh,estimated_total_co2_kg,estimated_total_co2_g
0,OK,False,True,0,NaN,NaN,NaN,NaN,LLM-only,baseline,...,0.189775,0.000949,0.000043,0.042699,11363.698261,189.394971,3.156583,0.946975,0.042614,42.613868
1,OK,False,True,0,NaN,NaN,NaN,NaN,LLM-only,baseline,...,0.248965,0.001245,0.000056,0.056017,6617.495126,110.291585,1.838193,0.551458,0.024816,24.815607
2,OK,False,True,0,NaN,NaN,NaN,NaN,LLM-only,baseline,...,0.004558,0.000023,0.000001,0.001025,264.434777,4.407246,0.073454,0.022036,0.000992,0.991630
3,OK,False,True,0,NaN,NaN,NaN,NaN,LLM-only,baseline,...,0.258039,0.001290,0.000058,0.058059,54435.960198,907.266003,15.121100,4.536330,0.204135,204.134851
4,OK,False,True,0,NaN,NaN,NaN,NaN,LLM-only,baseline,...,0.074448,0.000372,0.000017,0.016751,4824.209151,80.403486,1.340058,0.402017,0.018091,18.090784


In [10]:
cols = [
    'status', 'method_family', 'method', 'dataset', 'n_eligible_documents',
    'sample_index_effective', 'attempts', 'elapsed_seconds_doc',
    'estimated_total_minutes', 'estimated_total_co2_g',
    'error_type', 'error_message'
]
df[[c for c in cols if c in df.columns]]

,status,method_family,method,dataset,n_eligible_documents,sample_index_effective,attempts,elapsed_seconds_doc,estimated_total_minutes,estimated_total_co2_g,error_type,error_message
0,OK,LLM-only,baseline,docred_causal,998,0,1,11.386471,189.394971,42.613868,NaN,NaN
1,OK,LLM-only,baseline,eventstoryline,443,0,1,14.937912,110.291585,24.815607,NaN,NaN
2,OK,LLM-only,baseline,fincausal,967,0,1,0.273459,4.407246,0.991630,NaN,NaN
3,OK,LLM-only,baseline,maven_ere,3516,0,1,15.482355,907.266003,204.134851,NaN,NaN
4,OK,LLM-only,baseline,causalbank,1080,0,1,4.466860,80.403486,18.090784,NaN,NaN
5,OK,LLM-only,cot,docred_causal,998,0,1,50.098290,833.301549,187.492849,NaN,NaN
6,OK,LLM-only,cot,eventstoryline,443,0,1,57.327473,423.267845,95.235265,NaN,NaN
7,OK,LLM-only,cot,fincausal,967,0,1,10.716403,172.712695,38.860356,NaN,NaN
8,OK,LLM-only,cot,causalbank,1080,0,1,13.809294,248.567286,55.927639,NaN,NaN
9,OK,LLM-only,cot,maven_ere,3516,0,1,63.636531,3729.100716,839.047661,NaN,NaN
